# Digit Recognizer — 작게/빠르게 배우는 '생성 → 인식 검증' (Colab)

**연습 기법** (IOAI 2024 *Madarian Cow* 의 핵심 패턴): **생성모델로 목표 클래스 이미지를 만들고, 인식모델이 맞게
인식하는지로 검증**한다. 여기서는 **28×28 작은 숫자**라 몇 초 만에 학습·확인할 수 있다.

**시나리오 매핑**:
- Madarian Cow: 디퓨전으로 '소/고양이…' 생성 → **DETR** 가 올바른 객체로 검출하면 정답.
- 여기: **conditional VAE** 로 '숫자 d' 생성 → **분류기**가 d 로 인식하면 정답. *인식률* 이 점수(높을수록 좋음).

코드는 `torch` 기본만. **보너스로 분류기 예측을 Digit Recognizer 에 제출**(상시 제출 가능)할 수 있다.


## 0. 라이브러리 설치 & Kaggle 자격증명


In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")


In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")


## 1. Kaggle 에서 Digit Recognizer (MNIST 28×28) 다운로드


In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("digit-recognizer", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])


## 2. 데이터 로딩


In [ ]:
import os, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"

train = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
X = torch.tensor(train.drop(columns=["label"]).values, dtype=torch.float32).div(255.).to(device)  # (N,784)
y = torch.tensor(train["label"].values, dtype=torch.long).to(device)
X_test = torch.tensor(test.values, dtype=torch.float32).div(255.).to(device)
def onehot(c): return F.one_hot(c, 10).float()
print("train:", X.shape, " test:", X_test.shape)


## 3. 인식모델 (분류기) — Madarian Cow 의 DETR 역할
생성된 숫자가 '맞는 숫자'인지 판정하는 모델. (제출에도 재사용)


In [ ]:
clf = nn.Sequential(nn.Linear(784,128), nn.ReLU(), nn.Linear(128,10)).to(device)
opt = torch.optim.Adam(clf.parameters(), 1e-3)
for ep in range(3):
    perm = torch.randperm(len(X))
    for i in range(0, len(X), 256):
        b = perm[i:i+256]; opt.zero_grad()
        F.cross_entropy(clf(X[b]), y[b]).backward(); opt.step()
print("분류기 학습 완료 — train accuracy:", round((clf(X).argmax(1)==y).float().mean().item(), 3))


## 4. 생성모델 (conditional VAE) — Madarian Cow 의 디퓨전 역할
클래스를 조건으로 받아 그 숫자 이미지를 생성. 작아서 몇 초면 학습된다.


In [ ]:
ZD = 20
class CVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(794,400), nn.ReLU())
        self.mu, self.lv = nn.Linear(400,ZD), nn.Linear(400,ZD)
        self.dec = nn.Sequential(nn.Linear(ZD+10,400), nn.ReLU(), nn.Linear(400,784), nn.Sigmoid())
    def forward(self, x, c):
        h = self.enc(torch.cat([x,c],1)); mu, lv = self.mu(h), self.lv(h)
        z = mu + torch.randn_like(mu)*torch.exp(0.5*lv)
        return self.dec(torch.cat([z,c],1)), mu, lv
    def generate(self, c):                          # 사전분포에서 샘플 → 클래스 조건으로 생성
        z = torch.randn(len(c), ZD, device=device)
        return self.dec(torch.cat([z,c],1))

vae = CVAE().to(device); opt = torch.optim.Adam(vae.parameters(), 1e-3)
for ep in range(15):
    perm = torch.randperm(len(X))
    for i in range(0, len(X), 256):
        b = perm[i:i+256]; xb = X[b]; cb = onehot(y[b])
        xr, mu, lv = vae(xb, cb); opt.zero_grad()
        rec = F.binary_cross_entropy(xr, xb, reduction="sum")/len(b)
        kl  = -0.5*torch.sum(1+lv-mu.pow(2)-lv.exp())/len(b)
        (rec+kl).backward(); opt.step()
print("conditional VAE 학습 완료")


## 5. 생성 → 인식 검증 (Madarian Cow 패턴)
각 숫자를 생성해 분류기가 맞게 인식하는 비율(*인식률*)을 점수로 본다.


In [ ]:
import matplotlib.pyplot as plt
vae.eval()
with torch.no_grad():
    total = 0; samples = []
    for d in range(10):
        c = onehot(torch.full((100,), d, device=device))
        gen = vae.generate(c)
        rate = (clf(gen).argmax(1) == d).float().mean().item(); total += rate
        samples.append(gen[0].cpu().view(28,28))
        print(f"  숫자 {d}: 인식률 {rate:.2f}")
    print(f"\n전체 인식률(생성→분류기) = {total/10:.3f}  (높을수록 좋음 — Madarian Cow 의 생성→검출과 동일)")
# 생성 샘플 미리보기
fig, ax = plt.subplots(1, 10, figsize=(12, 1.5))
for d in range(10):
    ax[d].imshow(samples[d], cmap="gray"); ax[d].axis("off"); ax[d].set_title(str(d))
plt.show()


## 6. (보너스) Digit Recognizer 제출 파일
위 분류기로 test 를 예측해 `submission.csv` 생성 → 상시 제출 가능.


In [ ]:
clf.eval()
with torch.no_grad():
    pred = clf(X_test).argmax(1).cpu().numpy()
submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"ImageId": np.arange(1, len(pred)+1), "Label": pred}).to_csv(submission_path, index=False)
print("Saved:", submission_path); pd.read_csv(submission_path).head()


In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)


## 🏁 정리
- **핵심 연습**: conditional VAE 로 숫자 생성 → 분류기로 인식 검증 = IOAI *Madarian Cow* 의 *생성→인식* 패턴(작고 빠른 버전).
- **보너스**: `submission.csv` 를 [Digit Recognizer](https://www.kaggle.com/competitions/digit-recognizer) 에 제출(상시 가능).

인식률을 올리려면: VAE 학습 epoch 늘리기, 잠재차원 `ZD` 조정, 더 좋은 생성모델(DCGAN/디퓨전). Madarian Cow 처럼 '생성 조건을 조정해 인식모델을 만족시키는' 연습을 작은 데이터로 빠르게 반복할 수 있다.
